# 31. UI 시안 C 학습 노트 — 답변·근거 두 칸 + 테마 (2026-11-09)

시안 C의 핵심은 "답변(왼쪽)과 근거 문서(오른쪽)를 나란히" 보여 주는 것이다. 실제 코드는 `src/rag/ui/app.py`와 `.streamlit/config.toml`에 있고, 이 노트는 같은 내용을 주석과 함께 정리한 학습 자료다.

## 원복 방법
| 방법 | 명령 |
|---|---|
| 로컬 태그 | `git checkout pre-design-20261109 -- src/rag/ui/app.py` 그리고 `.streamlit/config.toml` 삭제 (태그는 예시 질문·대화 지우기까지 반영된 상태, 커밋 2609701) |
| 백업 파일 | `notebooks/prevention_legal/backups/app_before_design_20261109.py` (Git 제외, 이 컴퓨터에만 있음) |
| 테마만 끄기 | `.streamlit/config.toml` 삭제. CSS만 끄기: `app.py`의 `st.markdown(PAGE_CSS, ...)` 한 줄 삭제 |

시험 `test_domain_answer_is_split_into_answer_and_sources_columns` 등은 원복하면 실패하므로 함께 뺀다.

## 이 노트에서 배우는 것
1. 색·글꼴은 **테마 파일**(`.streamlit/config.toml`), 테마로 안 되는 것만 **CSS 한 덩어리**로 한다.
2. `st.columns`로 두 칸을 만들고, 칸마다 `with` 블록 안에서 그린다. 좁은 화면에서는 Streamlit이 위아래로 쌓는다.
3. `st.container(border=True)`로 테두리 카드를 만든다(CSS 없이).
4. 사용자 입력이 섞이지 않는 고정 문구만 `unsafe_allow_html=True`로 그리고, 그래도 `html.escape`를 한다.

In [ ]:
# ── 1) .streamlit/config.toml (테마) ─────────────────────────────────
# 프로젝트 루트에서 `streamlit run src/rag/ui/app.py`를 실행하면 루트의 .streamlit/config.toml을 읽는다.
# Streamlit Community Cloud도 저장소 루트의 이 파일을 읽는다.
CONFIG_TOML = """
[theme]
base = "light"                      # 밝은 테마로 고정(어두운 모드 대응은 하지 않는다)
primaryColor = "#2457d6"            # 버튼 강조·탭 밑줄 등 대표색
backgroundColor = "#eef1f5"         # 페이지 배경(연한 회청색)
secondaryBackgroundColor = "#ffffff"  # 입력창·사이드바 배경(흰색)
textColor = "#18212d"               # 본문 글자색(검정에 가까운 남색)
font = "sans serif"
"""
# 주의: 접속 정보·비밀번호는 여기에 넣지 않는다. .streamlit/secrets.toml 은 .gitignore 에 넣어 두었다.
print(CONFIG_TOML)

In [ ]:
# ── 2) 테마로 안 되는 것만 CSS (app.py 의 PAGE_CSS) ─────────────────
PAGE_CSS = """
<style>
/* 글꼴: IBM Plex Sans KR. 못 불러오면(오프라인) 뒤의 대체 글꼴로 보인다. 화면 기능에는 영향 없음 */
@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Sans+KR:wght@400;500;600&display=swap');
.stApp, .stMarkdown, .stCaption, .stChatInput textarea, .stButton button { font-family: "IBM Plex Sans KR", "Malgun Gothic", sans-serif; }

/* 제목은 조금 가늘게, 자간은 살짝 좁게 */
h1 { font-weight: 600; letter-spacing: -0.01em; }

/* 칸 제목(답변 / 근거 문서): 작은 글씨 + 자간 + 회색, 오른쪽 끝에 건수 */
.panel-title { display: flex; justify-content: space-between; font-size: 0.78rem; font-weight: 600;
               letter-spacing: 0.06em; color: #5b6676; margin: 0 0 0.35rem 0; }

/* 큰 박스(답변·근거)는 흰색, 박스 안의 근거 카드는 아주 연한 회청색. 이름표(st-key-…)는 코드의 container key 와 짝이다.
   (처음에는 data-testid="stVerticalBlockBorderWrapper" 로 지정했지만 설치된 Streamlit 1.67에는 그 이름이 없어 흰색이 적용되지 않았다.
    그래서 container 에 key 를 주고 .st-key-<key> 클래스로 지정한다. 내부 구조 이름보다 key 가 버전이 바뀌어도 덜 깨진다.) */
[class*="st-key-answerbox-"], [class*="st-key-sourcebox-"] { background: #ffffff; border-radius: 0.5rem; }
[class*="st-key-doccard-"] { background: #f4f6fa; }

/* 상태 배지(답변 완료·일부 확인 등): 칸 제목 오른쪽에 색 배경 글자. 클래스 이름은 코드의 TONES 안의 값만 쓴다 */
.tone { color: #ffffff; padding: 2px 10px; border-radius: 4px; letter-spacing: 0; font-size: 0.8rem; }
.tone-green { background: #167a52; } .tone-orange { background: #b45f06; }
.tone-gray { background: #5b6676; }  .tone-red { background: #b3261e; }
</style>
"""
# main() 에서 한 번: st.markdown(PAGE_CSS, unsafe_allow_html=True)
# 이 CSS 를 지우면 기본 모양이 되므로, 문제가 생기면 이 줄만 빼면 된다.

## 3) 두 칸 배치 (`show_result`)
```
감염병(DOMAIN) 질문  → 왼쪽 답변 칸 | 오른쪽 근거 문서 칸
일반 안내·범위 밖     → 답변 칸만 (근거 칸이 없다)
```
기존 `show_user_notes`는 DOMAIN이 아니면 아무것도 그리지 않았다. 그 규칙을 그대로 두고, 칸 자체를 만들지 않는 조건을 같은 기준으로 맞췄다.

In [ ]:
import html
try:
    import streamlit as st  # 노트북 학습용 예시다. 실제 실행은 app.py 안에서 된다.
except ImportError:
    st = None  # streamlit 이 없는 커널에서는 함수 정의만 읽어 볼 수 있다(호출하지 않는다)


def panel_title(title: str, extra: str = "") -> None:
    """칸 제목. 고정 문구만 받으므로 HTML 로 그리되, 그래도 html.escape 로 이스케이프한다."""
    tail = f"<span>{html.escape(extra)}</span>" if extra else ""
    st.markdown(f'<div class="panel-title"><span>{html.escape(title)}</span>{tail}</div>', unsafe_allow_html=True)


def show_answer_panel(result: dict, developer: bool = False) -> None:
    """왼쪽 답변 칸. 상태 배지 + 최종 답변. (실제 코드는 app.py 의 show_answer_panel)"""
    output = result["output"]
    # 상태는 칸 제목 오른쪽의 색 배지로 그린다(panel_title 의 tone). st.badge 는 쓰지 않는다.
    panel_title("답변", "답변 완료", tone="green")
    st.markdown(output["answer"])          # 답변 문장은 화면에서 새로 만들지 않고 QueryOutput 그대로 보여 준다


def show_result(result: dict, documents: dict, developer: bool = False) -> None:
    output = result["output"]
    if output.get("response_type") == "DOMAIN":
        # [1.15, 1] 은 두 칸의 너비 비율. gap="large" 는 칸 사이 간격.
        left, right = st.columns([1.15, 1], gap="large")
        # `with left, st.container(border=True):` — 칸 안에 테두리 박스(흰 배경)를 한 겹 더 두른다
        # key 는 CSS 가 박스를 찾는 이름표다. 한 화면에 결과가 여러 개일 수 있어 요청 번호(request_id)를 붙여 서로 다르게 한다.
        rid = result.get("request_id", "r")
        with left, st.container(border=True, key=f"answerbox-{rid}"):
            show_answer_panel(result, developer)
        with right, st.container(border=True, key=f"sourcebox-{rid}"):
            # show_user_notes: 근거 문서마다 st.container(border=True, key=f"doccard-…") 카드 한 장씩(박스 안의 작은 카드)
            ...
    else:
        with st.container(border=True, key=f"answerbox-{result.get('request_id', 'r')}"):
            show_answer_panel(result, developer)   # 근거 칸 없이 답변만

## 4) 근거 카드 (`show_user_notes` 안)
```python
panel_title("근거 문서", f"{len(lines)}건" if lines else "")
for line in lines:
    with st.container(border=True):   # 테두리 카드. CSS 없이도 카드처럼 보인다
        st.markdown(line)             # 문서명(링크)·판본·PDF 쪽수 한 줄
```
- 카드에 **[1], [2] 번호를 붙이지 않았다.** `source_lines`는 문서 단위로 묶은 목록이라, 답변 속 `[1][2]`(인용 번호)와 번호가 일치하지 않을 수 있다. 번호를 붙이면 잘못된 연결을 암시한다.
- 시안에 있던 "발췌 강조"는 구현하지 않았다. 일반 사용자에게 발췌를 보이지 않는다는 기존 결정과 다르기 때문이다(팀 확인이 필요한 사항).

## 5) 시험 (`tests/rag/ui/test_app_buttons.py`)
| 시험 | 확인하는 약속 |
|---|---|
| 두 칸 | DOMAIN 답변은 칸이 2개, 왼쪽에 답변·오른쪽에 `근거 문서` 제목 |
| 근거 칸 없음 | GENERAL 응답은 칸이 0개이고 답변은 그대로 나온다 |
| 개발자 화면 | 두 칸 아래에 `출처`·`처리 흐름`·`그래프 경로` 탭이 그대로 있다 |

## 6) 직접 해 보기
1. `[1.15, 1]`을 `[1, 1]`과 `[2, 1]`로 바꿔 보고 어느 비율이 읽기 좋은지 비교해 보자.
2. `PAGE_CSS`의 `.panel-title` 색을 바꾸면 화면의 어디가 변하는지 확인해 보자.
3. `.tone-green` 배경색을 바꾸고 '답변 완료' 배지 색이 달라지는 것을 확인해 보자. 이어서 `config.toml`의 `primaryColor`를 바꾸고 버튼·탭 색이 달라지는 것을 확인해 보자(서버를 다시 시작해야 한다).
4. (생각해 보기) `unsafe_allow_html=True`는 왜 위험할 수 있고, 이 코드는 왜 `html.escape`를 쓰는가? 사용자가 입력한 질문을 같은 방식으로 그리면 어떻게 될까?

## 7) 이번 변경의 한계 (사실만 기록)
- 실제 앱을 브라우저(Chrome, 폭 1280·400)에서 열어 예시 질문을 눌러 화면 확인을 했다. **개발자 화면(`?dev=1`)은 AppTest로만 확인했고 브라우저로는 보지 못했다.**
- `data-testid` CSS는 Streamlit 버전이 바뀌면 적용되지 않을 수 있다(그때는 모양만 달라진다).
- 글꼴은 외부(Google Fonts)에서 불러온다. 발표 장소에서 인터넷이 없으면 대체 글꼴로 보인다.
- 시안의 도장·발췌 강조는 구현하지 않았다.

## 8) 질문 문장 스타일과 근거 칩 (시안 C의 나머지, 사용자 요청으로 추가)
**질문 문장** — 굵은 글씨 + 아래 검정 줄.
- 질문은 **사용자 입력**이다. HTML로 그리면 `<script>` 같은 글자가 섞일 수 있으므로 일반 마크다운(`st.markdown(question)`)으로 그리고, 스타일은 컨테이너 `key`로 붙는 CSS 클래스(`.st-key-qline-…`)가 한다.
- 시안처럼 아바타 없이 한 줄로 보이도록 `st.chat_message("user")` 대신 `st.container(key=...)`를 쓴다.
- 아래 줄은 컨테이너가 아니라 문단(`p`)에 그었다. 컨테이너에는 Streamlit이 먼저 정한 여백이 있어 줄이 글자에 붙어 보였기 때문이다(실제 화면으로 확인하고 고쳤다).

**근거 칩** — 문서명 아래에 발행처·판본·PDF 쪽·인쇄 쪽을 작은 칩으로 보여 준다.
- 칩 글자는 `sources.source_cards()`가 만든다(DB·manifest 메타데이터, 사용자 입력 아님). 그래도 화면에서 `html.escape`를 한다.
- 발행처가 `확인 필요`인 문서는 그 칩만 생략한다. 판본이 없으면 판본 칩도 없다.
- 칩 CSS 글꼴은 고정폭(IBM Plex Mono)이라 쪽수가 출처 표시처럼 보인다.
- 시험: `tests/rag/ui/test_sources.py`의 `source_cards` 시험 3개, `test_app_buttons.py`의 칩·질문 문장 시험.

In [ ]:
# ── 질문 문장 (app.py 의 question_line) ─────────────────────────────
def question_line(question, key):
    """질문 문장. 사용자 입력이므로 HTML 이 아니라 일반 마크다운으로 그린다."""
    with st.container(key=f"qline-{key}"):   # key 가 CSS 이름표(.st-key-qline-…)가 된다
        st.markdown(question)

QUESTION_CSS = """
[class*="st-key-qline-"] p { font-size: 1.1rem; font-weight: 600; margin: 0; padding-bottom: 0.55rem; border-bottom: 2px solid #18212d; }
"""

# ── 근거 카드 재료 (sources.py 의 source_cards 를 흉내 낸 예시) ──────────
def source_cards_example(title, issuer, version, pages, printed=()):
    """한 문서의 카드 재료: 굵은 문서명 + 칩 목록. 실제 함수는 sources.source_cards."""
    chips = [issuer, f"판본 {version}", "PDF " + ", ".join(f"{p}쪽" for p in pages)]
    if printed:
        chips.append("인쇄 " + ", ".join(f"{p}쪽" for p in printed))
    return {"title_md": f"**{title}**", "chips": chips}

print(source_cards_example("경기도 학교 감염병 예방관리 실무 가이드북", "경기도교육청", "2024.05", [393], [383]))
# 화면에서는 칩마다 <span class="src-chip">…</span> 로 그린다(글자는 html.escape 한 뒤).